In [1]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

Static Model

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    # other params...
)

c:\Users\ajiteshgowlikar\Desktop\langchain_projects\seminar\venv\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
messages = [
    (
        "system",
        "You are a helpful assistant that translates English to French. Translate the user sentence.",
    ),
    ("human", "I love programming."),
]
ai_msg = llm.invoke(messages)
ai_msg

AIMessage(content="J'adore la programmation.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--52972d24-ee36-4a6e-80e9-c363bcfc771e-0', usage_metadata={'input_tokens': 21, 'output_tokens': 7, 'total_tokens': 28, 'input_token_details': {'cache_read': 0}})

In [4]:
print(ai_msg.content)

J'adore la programmation.


Dynamic model

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

# Initialize Gemini models
basic_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")  # Fast and cheaper
advanced_model = ChatGoogleGenerativeAI(model="gemini-2.5-pro")  # More capable

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose Gemini model dynamically based on conversation complexity."""
    message_count = len(request.state["messages"])

    # Switch to advanced model if the conversation is long or complex
    if message_count > 10:
        model = advanced_model
    else:
        model = basic_model

    request.model = model
    return handler(request)

# Define your tools (example placeholder)
tools = []  # Add your actual tool list here, e.g., search tools, database connectors, etc.

# Create agent with middleware
agent = create_agent(
    model=basic_model,  # Default model
    tools=tools,
    middleware=[dynamic_model_selection]
)

print("Dynamic Gemini Agent initialized successfully.")


Dynamic Gemini Agent initialized successfully.


In [6]:
result = agent.invoke({"messages": [HumanMessage(content="What is the weather in hyderabad today?")]})

# Print raw and parsed outputs
print(result)
if "messages" in result and len(result["messages"]) > 1:
    ai_message = result["messages"][-1]
    print("Answer:", ai_message.content)
else:
    print("No AI message found.")

{'messages': [HumanMessage(content='What is the weather in hyderabad today?', additional_kwargs={}, response_metadata={}, id='9265b369-6cda-40d4-9f85-244fd5702ca8'), AIMessage(content='I cannot provide real-time, live weather updates as I don\'t have access to current conditions.\n\nTo get the most accurate and up-to-date weather for Hyderabad right now, I recommend checking a reliable weather source such as:\n\n*   **Google Search:** Simply type "weather in Hyderabad"\n*   **Weather apps:** AccuWeather, The Weather Channel, etc.\n*   **Local news websites**\n\nThese sources will give you the current temperature, humidity, wind, and forecast for today.', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--d60d7f6a-928a-4dd2-8c7c-b8da25c88678-0', usage_metadata={'input_tokens': 10, 'output_tokens': 667, 'total_

Tools & simple Agent

In [14]:
import requests
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
@tool
def search(query: str) -> str:
    """Search for information on the web (placeholder demo)."""
    return f"Search results for: {query}"

@tool
def get_geo_coordinates(city: str) -> str:
    """Get latitude and longitude for a given city using Open-Meteo Geocoding API."""
    try:
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}"
        geo_resp = requests.get(geo_url, timeout=10)
        geo_data = geo_resp.json()

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return f"Could not find location: {city}"

        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]
        return f"Coordinates of {city}: Latitude {lat}, Longitude {lon}"

    except Exception as e:
        return f"Error fetching coordinates for {city}: {str(e)}"

@tool
def get_weather(location: str) -> str:
    """Get real-time weather information for a given city using Open-Meteo API."""
    try:
        # Step 1: Geocode the city → get latitude & longitude
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}"
        geo_resp = requests.get(geo_url, timeout=10)
        geo_data = geo_resp.json()

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            return f"Could not find location: {location}"

        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]

        # Step 2: Fetch real-time weather data
        weather_url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}&current_weather=true"
        )
        weather_resp = requests.get(weather_url, timeout=10)
        weather_data = weather_resp.json()

        temp = weather_data["current_weather"]["temperature"]
        wind = weather_data["current_weather"]["windspeed"]
        desc = weather_data["current_weather"].get("weathercode", "Clear")

        return f"Current weather in {location}: {temp}°C, wind speed {wind} km/h, condition code {desc}"

    except Exception as e:
        return f"Error fetching weather for {location}: {str(e)}"

# Create the Gemini agent with both tools
agent = create_agent(
    model=llm,
    tools=[search, get_weather, get_geo_coordinates],
)

context = input("Enter your question about the weather: ")
# Query the agent
result = agent.invoke({"messages": [HumanMessage(content=context)]})

# Extract the AI message response
if "messages" in result and len(result["messages"]) > 1:
    ai_message = result["messages"][-1]
    print("Agent:", ai_message.content)
else:
    print("No AI message found.")


c:\Users\ajiteshgowlikar\Desktop\langchain_projects\seminar\venv\lib\site-packages\langchain_google_genai\chat_models.py:2316: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


ValueError: No content messages found. The Gemini API requires at least one non-system message (HumanMessage, AIMessage, etc.) in addition to any SystemMessage. Please include additional messages in your input.

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain_core.messages import ToolMessage

tool
def search(query: str) -> str:
    """Search for information on the web (placeholder demo)."""
    return f"Search results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get real-time weather information for a given city using Open-Meteo API."""
    try:
        # Step 1: Geocode the city → get latitude & longitude
        geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={location}"
        geo_resp = requests.get(geo_url, timeout=10)
        geo_data = geo_resp.json()

        # if "results" not in geo_data or len(geo_data["results"]) == 0:
        #     return f"Could not find location: {location}"

        if "results" not in geo_data or len(geo_data["results"]) == 0:
            raise ValueError(f"Could not find location: {location}")
        lat = geo_data["results"][0]["latitude"]
        lon = geo_data["results"][0]["longitude"]

        # Step 2: Fetch real-time weather data
        weather_url = (
            f"https://api.open-meteo.com/v1/forecast?"
            f"latitude={lat}&longitude={lon}&current_weather=true"
        )
        weather_resp = requests.get(weather_url, timeout=10)
        weather_data = weather_resp.json()

        temp = weather_data["current_weather"]["temperature"]
        wind = weather_data["current_weather"]["windspeed"]
        desc = weather_data["current_weather"].get("weathercode", "Clear")


        return f"Current weather in {location}: {temp}°C, wind speed {wind} km/h, condition code {desc}"

    except Exception as e:
        return f"Error fetching weather for {location}: {str(e)}"
    
@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

agent = create_agent(
    model=llm,
    tools=[search, get_weather],
    middleware=[handle_tool_errors]
)


result = agent.invoke({"messages": [HumanMessage(content="Give me the weather in UnknownCity right now.")]})


if "messages" in result and len(result["messages"]) > 1:
    ai_message = result["messages"][-1]
    print("Agent:", ai_message.content)
else:
    print("No AI message found.")


Agent: [{'type': 'text', 'text': 'I can\'t seem to find "UnknownCity" in my system. Could you please provide a valid city?', 'extras': {'signature': 'Cs0CAdHtim8leB30HOEJrXBep7rTGmepZEQdVxJIVs3VYAf3siAxbpZx5Gb6n20+88f7wIdWbiRYwlJnH43rb+Y/QpZxnmyDzVCj7FgXEzrwEwOAIzenxx7Sj9TCYhDIvsT8+k6jYDTmVd5/LDJ7/izjmylB0Nprzt7aNMop0O5LoG4Zj4CgO3ngzp4Cn4odfetZgLhcy6Fe+ANYSORabEoo7VyC3kUvUmUGGY4uzpGcyobDqpYkpe5Yl0S0Sdw+8F+G1rLR42tb4lFmlB7RyHSOOzkt/U0SgrZKsrVgFXuDGBE+2juOeLbK0Bi3vQrvT+OY7pWaUvGGkkOYMM2oPHwi80fIgqlT5g3XTO716X0uLG2avO9xeyRqjQXWuhBqYDAoOmQIXSYM1N1zFtIcKfzX4qA2wrF/YX9jNwb6W4HLKRZs6CVnCzx2a7BrlU7G'}}]


System Prompts

In [9]:
agent = create_agent(
    llm,
    tools,
    system_prompt="You are a helpful assistant. Be concise and accurate."
)

Dynamic System Prompts


In [10]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model=llm,
    tools=[search],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)

Invocation

In [11]:
result = agent.invoke(
    {"messages": [HumanMessage(content="What's the weather in San Francisco?")]},
    context={"user_role": "user"}
)

-------------------------------------- Advanced Concepts --------------------------------------------

Structured output

ToolStrategy

In [12]:
from pydantic import BaseModel
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

agent = create_agent(
    model=llm,
    tools=[search],
    response_format=ToolStrategy(ContactInfo)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

Provider Strategy

In [13]:
from pydantic import BaseModel
from langchain.agents import create_agent, ProviderStrategy
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Define structured schema
class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

# 2. Build Gemini model with strict JSON output
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    temperature=0,
    response_mime_type="application/json",   # ← REQUIRED FOR GEMINI JSON OUTPUT
)

# 3. Create the agent (correct usage)
agent = create_agent(
    model=llm,
    response_format=ProviderStrategy(ContactInfo)
)

# 4. Invoke the agent
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

# 5. Get structured output
print(result["structured_response"])


ImportError: cannot import name 'ProviderStrategy' from 'langchain.agents' (c:\Users\ajiteshgowlikar\Desktop\langchain_projects\seminar\venv\lib\site-packages\langchain\agents\__init__.py)

Memory

Defining state via middleware

In [ ]:
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any


class CustomState(AgentState):
    user_preferences: dict

class CustomMiddleware(AgentMiddleware):
    state_schema = CustomState
    tools = [tool1, tool2]

    def before_model(self, state: CustomState, runtime) -> dict[str, Any] | None:
        ...

agent = create_agent(
    model,
    tools=tools,
    middleware=[CustomMiddleware()]
)

# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

Defining state via state_schema

In [ ]:
from langchain.agents import AgentState


class CustomState(AgentState):
    user_preferences: dict

agent = create_agent(
    model,
    tools=[tool1, tool2],
    state_schema=CustomState
)
# The agent can now track additional state beyond messages
result = agent.invoke({
    "messages": [{"role": "user", "content": "I prefer technical explanations"}],
    "user_preferences": {"style": "technical", "verbosity": "detailed"},
})

Streaming

In [ ]:
for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

Agent: Search for AI news and summarize the findings


StructuredOutputValidationError: Failed to parse structured output for tool 'ContactInfo': Native structured output expected valid JSON for ContactInfo, but parsing failed: Expecting value: line 1 column 1 (char 0)..